# H&M Customer Preference Preprocessing

**Owner:** Binara  
**Module:** CCS4310 – Deep Learning  
**Project:** Explainable-Fashion-Design-AI

This notebook builds memory-efficient, chronological, leakage-safe interaction data and reusable profiles. It reads raw files but never modifies them. FAST_DEV_RUN is intentionally enabled for development.

In [1]:
from pathlib import Path
import json
import sys
import time
from time import perf_counter
from collections import Counter, defaultdict, deque
from tqdm.auto import tqdm
import numpy as np
import pandas as pd

RANDOM_STATE = 42
FAST_DEV_RUN = False
TRANSACTION_CHUNKSIZE = 250_000
MAX_TRANSACTIONS_DEV = 10_000
MAX_CUSTOMERS_DEV = 300
TRAIN_NEGATIVES_PER_POSITIVE = 2
NEGATIVE_SAMPLES_PER_POSITIVE = TRAIN_NEGATIVES_PER_POSITIVE
MIN_EVAL_CANDIDATES_PER_CUSTOMER = 30
RECENT_WINDOW_DAYS = 30
rng = np.random.default_rng(RANDOM_STATE)

def find_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p/'data/raw/hm').is_dir() and (p/'requirements.txt').exists(): return p
    raise FileNotFoundError('Repository root not found.')

def normalize_article_id(series):
    return series.astype('string').str.strip().str.zfill(10)

ROOT=find_root(); RAW_HM_DIR=ROOT/'data/raw/hm'; INTERIM_DIR=ROOT/'data/interim'; INTERIM_DIR.mkdir(parents=True, exist_ok=True)
ARTICLES_PATH=RAW_HM_DIR/'articles.csv'; CUSTOMERS_PATH=RAW_HM_DIR/'customers.csv'; TRANSACTIONS_PATH=RAW_HM_DIR/'transactions_train.csv'
print('Using existing shared repository .venv / selected project kernel:', sys.executable)

Using existing shared repository .venv / selected project kernel: d:\Deep Learning\Project\GITHUB\Explainable-Fashion-Design-AI\.venv\Scripts\python.exe


In [2]:
article_features=pd.read_csv(ARTICLES_PATH, dtype={'article_id':'string'})
article_features['article_id']=normalize_article_id(article_features['article_id'])
customers=pd.read_csv(CUSTOMERS_PATH, usecols=['customer_id'], dtype={'customer_id':'string'})
usecols=['t_dat','customer_id','article_id','price','sales_channel_id']
dtype={'customer_id':'string','article_id':'string','price':'float32','sales_channel_id':'int8'}
if FAST_DEV_RUN:
    parts=[]; loaded=0
    for ch in pd.read_csv(TRANSACTIONS_PATH,usecols=usecols,dtype=dtype,chunksize=TRANSACTION_CHUNKSIZE):
        ch['article_id']=normalize_article_id(ch['article_id'])
        parts.append(ch); loaded += len(ch)
        if loaded >= TRANSACTION_CHUNKSIZE: break
    tx=pd.concat(parts,ignore_index=True)
else:
    tx=pd.read_csv(TRANSACTIONS_PATH,usecols=usecols,dtype=dtype)
tx['t_dat']=pd.to_datetime(tx['t_dat'],errors='coerce')
if FAST_DEV_RUN:
    available_dates=np.sort(tx['t_dat'].dropna().unique())
    if len(available_dates) < 3:
        raise ValueError('FAST_DEV_RUN requires at least three distinct transaction dates.')
    date_positions=np.linspace(0, len(available_dates)-1, num=min(10, len(available_dates)), dtype=int)
    selected_dates=available_dates[np.unique(date_positions)]
    per_date_limit=max(1, MAX_TRANSACTIONS_DEV // len(selected_dates))
    tx=(tx[tx['t_dat'].isin(selected_dates)]
         .sort_values(['t_dat','customer_id','article_id'])
         .groupby('t_dat', sort=True, group_keys=False)
         .head(per_date_limit)
         .head(MAX_TRANSACTIONS_DEV)
         .copy())
else:
    tx=tx.head(MAX_TRANSACTIONS_DEV if FAST_DEV_RUN else None)
invalid_before=len(tx)
tx=tx.dropna(subset=['t_dat','customer_id','article_id']).query('price >= 0').drop_duplicates().copy()
if FAST_DEV_RUN:
    keep=tx['customer_id'].drop_duplicates().head(MAX_CUSTOMERS_DEV); tx=tx[tx['customer_id'].isin(keep)].copy()
print('Loaded valid development rows:',len(tx),'date range:',tx.t_dat.min(),tx.t_dat.max())
print('Distinct transaction dates retained:',tx['t_dat'].nunique())
print('Dropped invalid/duplicate rows:', invalid_before-len(tx))
for c, fn in [('year',lambda x:x.dt.year),('month',lambda x:x.dt.month),('week',lambda x:x.dt.isocalendar().week.astype('int16')),('day_of_week',lambda x:x.dt.dayofweek)]: tx[c]=fn(tx['t_dat'])

Loaded valid development rows: 28813419 date range: 2018-09-20 00:00:00 2020-09-22 00:00:00
Distinct transaction dates retained: 734
Dropped invalid/duplicate rows: 2974905


In [3]:
# Strict chronological 70/15/15 split. Every snapshot below is built only from history before its event/window.
tx=tx.sort_values(['t_dat','customer_id','article_id']).reset_index(drop=True)
dates=np.sort(tx['t_dat'].dropna().unique())
if len(dates)<3: raise ValueError('Not enough distinct dates for chronological split.')
train_end=pd.Timestamp(dates[max(0,int(len(dates)*.70)-1)]); valid_end=pd.Timestamp(dates[max(0,int(len(dates)*.85)-1)])
train=tx[tx.t_dat<=train_end].copy(); valid=tx[(tx.t_dat>train_end)&(tx.t_dat<=valid_end)].copy(); test=tx[tx.t_dat>valid_end].copy()
print('Train date range:', train.t_dat.min(), 'to', train.t_dat.max(), '| unique customers:', train.customer_id.nunique(), '| unique articles:', train.article_id.nunique())
print('Validation date range:', valid.t_dat.min(), 'to', valid.t_dat.max(), '| unique customers:', valid.customer_id.nunique(), '| unique articles:', valid.article_id.nunique())
print('Test date range:', test.t_dat.min(), 'to', test.t_dat.max(), '| unique customers:', test.customer_id.nunique(), '| unique articles:', test.article_id.nunique())
assert train.t_dat.max() < valid.t_dat.min() and valid.t_dat.max() < test.t_dat.min()
article_cols=['article_id','product_type_name','product_group_name','garment_group_name','department_name','section_name','colour_group_name','perceived_colour_value_name']
article_features=article_features[[c for c in article_cols if c in article_features.columns]].drop_duplicates('article_id').copy()
# Dynamic popularity and price features are filled from past-only snapshots during example generation.
for feature_name in ['historical_article_popularity','recent_article_popularity']:
    article_features[feature_name]=0.0
article_features['historical_article_price']=np.nan
article_features.to_csv(INTERIM_DIR/'hm_article_features.csv',index=False)
print('Article metadata match rate:', f"{tx.article_id.isin(set(article_features.article_id)).mean():.2%}")

Train date range: 2018-09-20 00:00:00 to 2020-02-14 00:00:00 | unique customers: 1153883 | unique articles: 83671
Validation date range: 2020-02-15 00:00:00 to 2020-06-03 00:00:00 | unique customers: 548527 | unique articles: 42177
Test date range: 2020-06-04 00:00:00 to 2020-09-22 00:00:00 | unique customers: 591464 | unique articles: 45626
Article metadata match rate: 100.00%


In [4]:
def as_set(values):
    if values is None:
        return set()
    return set(values)


def empty_state():
    return {"customers": {}}


def _customer_bucket(state, customer_id):
    return state["customers"].setdefault(
        customer_id,
        {
            "articles": set(),
            "purchase_count": 0,
            "price_sum": 0.0,
            "category_counts": Counter(),
            "garment_counts": Counter(),
            "colour_counts": Counter(),
            "last_purchase": None,
            "recent_dates": deque(),
        },
    )


def add_event(state, event, article_lookup):
    cid = event.customer_id

    bucket = _customer_bucket(state, cid)

    aid = event.article_id

    bucket["articles"].add(aid)

    bucket["purchase_count"] += 1

    if pd.notna(getattr(event, "price", np.nan)):
        bucket["price_sum"] += float(event.price)

    bucket["last_purchase"] = pd.Timestamp(event.t_dat)
    bucket["recent_dates"].append(bucket["last_purchase"])

    meta = article_lookup.get(aid, {})

    for key, counter in (
        ("product_group_name", bucket["category_counts"]),
        ("garment_group_name", bucket["garment_counts"]),
        ("colour_group_name", bucket["colour_counts"]),
    ):
        value = meta.get(key)

        if pd.notna(value):
            counter[value] += 1

    return bucket


def make_state(history, article_lookup):
    state = empty_state()

    ordered = history.sort_values(["t_dat", "customer_id", "article_id"])

    for event in ordered.itertuples(index=False):
        add_event(state, event, article_lookup)

    return state


def _article_lookup(article_features):
    return article_features.set_index("article_id").to_dict("index")


def _empty_article_state():
    return {
        "counts": Counter(),
        "price_sum": defaultdict(float),
        "price_count": Counter(),
        "recent_counts": Counter(),
        "recent_days": deque(),
    }


def _add_article_day(state, day, article_lookup):
    day_counts = Counter()
    day_price_sum = defaultdict(float)
    day_price_count = Counter()

    for event in day.itertuples(index=False):
        aid = event.article_id

        day_counts[aid] += 1

        state["counts"][aid] += 1

        if pd.notna(getattr(event, "price", np.nan)):
            value = float(event.price)

            state["price_sum"][aid] += value
            state["price_count"][aid] += 1

            day_price_sum[aid] += value
            day_price_count[aid] += 1

    state["recent_days"].append((pd.Timestamp(day["t_dat"].iloc[0]), day_counts, day_price_sum, day_price_count))
    state["recent_counts"].update(day_counts)


def _trim_article_days(state, date):
    cutoff = pd.Timestamp(date) - pd.Timedelta(days=RECENT_WINDOW_DAYS)

    while state["recent_days"] and state["recent_days"][0][0] < cutoff:
        _, old_counts, _, _ = state["recent_days"].popleft()
        state["recent_counts"].subtract(old_counts)
        state["recent_counts"] += Counter()


def _article_dynamic_features(state, aid):
    count = state["counts"].get(aid, 0)
    price_count = state["price_count"].get(aid, 0)

    return {
        "historical_article_popularity": float(count),
        "recent_article_popularity": float(state["recent_counts"].get(aid, 0)),
        "historical_article_price": state["price_sum"].get(aid, 0.0) / price_count if price_count else np.nan,
    }


def _article_state_before(history, cutoff, article_lookup):
    state = _empty_article_state()

    ordered = history[history["t_dat"] < pd.Timestamp(cutoff)].sort_values(["t_dat", "article_id"])

    for _, day in ordered.groupby("t_dat", sort=True):
        _trim_article_days(state, day["t_dat"].iloc[0])
        _add_article_day(state, day, article_lookup)

    _trim_article_days(state, cutoff)

    return state


def _trim_recent_dates(bucket, date):
    cutoff = pd.Timestamp(date) - pd.Timedelta(days=RECENT_WINDOW_DAYS)

    recent_dates = bucket["recent_dates"]

    while recent_dates and recent_dates[0] < cutoff:
        recent_dates.popleft()


def _customer_features(state, cid, date):
    bucket = state["customers"].get(cid)

    if bucket is None:
        return {
            "past_purchase_count": 0,
            "recent_purchase_count": 0,
            "historical_average_price": np.nan,
            "unique_articles_purchased": 0,
            "unique_categories_purchased": 0,
            "days_since_previous_purchase": np.nan,
            "preferred_category": None,
            "preferred_garment_group": None,
            "preferred_colour": None,
        }

    _trim_recent_dates(bucket, date)

    last_purchase = bucket["last_purchase"]

    return {
        "past_purchase_count": bucket["purchase_count"],
        "recent_purchase_count": len(bucket["recent_dates"]),
        "historical_average_price": bucket["price_sum"] / bucket["purchase_count"] if bucket["purchase_count"] else np.nan,
        "unique_articles_purchased": len(bucket["articles"]),
        "unique_categories_purchased": len(bucket["category_counts"]),
        "days_since_previous_purchase": (pd.Timestamp(date) - last_purchase).days if last_purchase is not None else np.nan,
        "preferred_category": bucket["category_counts"].most_common(1)[0][0] if bucket["category_counts"] else None,
        "preferred_garment_group": bucket["garment_counts"].most_common(1)[0][0] if bucket["garment_counts"] else None,
        "preferred_colour": bucket["colour_counts"].most_common(1)[0][0] if bucket["colour_counts"] else None,
    }


def _candidate_row(customer_features, cid, aid, date, target, split_name, article_lookup, article_columns, dynamic_features):
    row = {
        "customer_id": cid,
        "article_id": aid,
        "t_dat": pd.Timestamp(date),
        "target": int(target),
        "split_name": split_name,
    }

    row.update(customer_features)

    meta = article_lookup.get(aid, {})

    for col in article_columns:
        row[col] = meta.get(col, np.nan)

    row.update(_article_dynamic_features(dynamic_features, aid))

    return row


def _sample_unseen(catalog, excluded, size):
    if size <= 0:
        return np.empty(0, dtype=object)

    selected = []
    selected_set = set()

    attempts = 0
    max_attempts = max(size * 20, 100)

    while len(selected) < size and attempts < max_attempts:
        aid = catalog[int(rng.integers(0, len(catalog)))]

        attempts += 1

        if aid not in excluded and aid not in selected_set:
            selected.append(aid)
            selected_set.add(aid)

    if len(selected) < size:
        available = catalog[~np.isin(catalog, list(excluded.union(selected_set)))]

        if len(available):
            extra = rng.choice(available, size=min(size - len(selected), len(available)), replace=False)
            selected.extend(extra.tolist())

    return np.asarray(selected, dtype=object)


def build_train_examples(events, article_features):
    started = time.perf_counter()

    article_lookup = _article_lookup(article_features)
    article_columns = [c for c in article_features.columns if c != "article_id"]
    catalog = article_features["article_id"].dropna().astype("string").unique()

    customer_state = empty_state()
    article_state = _empty_article_state()

    output_path = INTERIM_DIR / "hm_preference_train.csv"
    if output_path.exists():
        output_path.unlink()

    row_buffer = []
    batch_size = 10_000
    output_started = False
    row_count = 0
    columns = []

    def flush_buffer():
        nonlocal output_started, row_count, columns

        if row_buffer:
            batch = pd.DataFrame(row_buffer)
            batch["article_id"] = batch["article_id"].astype("string").str.strip().str.zfill(10)
            batch["t_dat"] = pd.to_datetime(batch["t_dat"], errors="raise")
            batch.to_csv(output_path, mode="a", header=not output_started, index=False)
            output_started = True
            row_count += len(batch)
            columns = list(batch.columns)
            row_buffer.clear()

    ordered = events.sort_values(["t_dat", "customer_id", "article_id"])

    for date, day in tqdm(ordered.groupby("t_dat", sort=True), total=ordered["t_dat"].nunique(), desc="Training dates"):
        _trim_article_days(article_state, date)

        day_by_customer = day.groupby("customer_id", sort=False)["article_id"].agg(set).to_dict()

        for event in day.itertuples(index=False):
            cid = event.customer_id

            customer_features = _customer_features(customer_state, cid, event.t_dat)
            row_buffer.append(_candidate_row(customer_features, cid, event.article_id, event.t_dat, 1, "train", article_lookup, article_columns, article_state))

            customer = customer_state["customers"].get(cid, {"articles": set()})
            excluded = set(customer.get("articles", set())).union(day_by_customer.get(cid, set()))

            for aid in _sample_unseen(catalog, excluded, TRAIN_NEGATIVES_PER_POSITIVE):
                row_buffer.append(_candidate_row(customer_features, cid, aid, event.t_dat, 0, "train", article_lookup, article_columns, article_state))

            if len(row_buffer) >= batch_size:
                flush_buffer()

        for event in day.itertuples(index=False):
            add_event(customer_state, event, article_lookup)

        _add_article_day(article_state, day, article_lookup)

    flush_buffer()

    print(f"Train examples built in {time.perf_counter() - started:.1f}s")

    return {
        "output_path": str(output_path),
        "row_count": row_count,
        "columns": columns,
    }


def _precompute_article_states(history, evaluation_dates, article_lookup):
    ordered = history.sort_values(["t_dat", "article_id"])
    history_days = ordered.groupby("t_dat", sort=True)
    evaluation_dates = sorted(pd.Timestamp(date) for date in evaluation_dates)
    article_state = _empty_article_state()

    history_day_iterator = iter(history_days)
    next_history_day = next(history_day_iterator, None)

    with tqdm(total=len(evaluation_dates), bar_format="Precomputing article states: {n_fmt}/{total_fmt} dates") as progress:
        for evaluation_date in evaluation_dates:
            while next_history_day is not None:
                history_date, day = next_history_day
                history_date = pd.Timestamp(history_date)

                if history_date >= evaluation_date:
                    break

                _trim_article_days(article_state, history_date)
                _add_article_day(article_state, day, article_lookup)
                next_history_day = next(history_day_iterator, None)

            _trim_article_days(article_state, evaluation_date)
            progress.update(1)
            yield evaluation_date, article_state


def build_eval_examples(history, events, split_name, article_features):
    started = time.perf_counter()

    article_lookup = _article_lookup(article_features)
    article_columns = [c for c in article_features.columns if c != "article_id"]
    catalog = article_features["article_id"].dropna().astype("string").unique()

    customer_state = make_state(history, article_lookup)
    evaluation_dates = pd.to_datetime(events["t_dat"].dropna().unique())
    article_states = _precompute_article_states(history, evaluation_dates, article_lookup)

    output_path = INTERIM_DIR / f"hm_preference_{split_name}.csv"
    if output_path.exists():
        output_path.unlink()

    row_buffer = []
    batch_size = 50_000
    output_started = False
    row_count = 0
    columns = []

    def flush_buffer():
        nonlocal output_started, row_count, columns

        if row_buffer:
            batch = pd.DataFrame(row_buffer)
            batch["article_id"] = batch["article_id"].astype("string").str.strip().str.zfill(10)
            batch["t_dat"] = pd.to_datetime(batch["t_dat"], errors="raise")
            batch.to_csv(output_path, mode="a", header=not output_started, index=False)
            output_started = True
            row_count += len(batch)
            columns = list(batch.columns)
            row_buffer.clear()

    customers_by_date = {}
    for cid, group in events.groupby("customer_id", sort=False):
        evaluation_date = pd.Timestamp(group["t_dat"].min())
        positives = as_set(group["article_id"])
        customers_by_date.setdefault(evaluation_date, []).append((cid, positives))

    customer_progress = tqdm(total=events["customer_id"].nunique(), desc=f"{split_name.capitalize()} customers")
    for date, article_state in article_states:
        for cid, positives in customers_by_date.get(date, []):
            customer = customer_state["customers"].get(cid, {"articles": set()})
            excluded = set(customer.get("articles", set())).union(positives)

            required_negatives = max(MIN_EVAL_CANDIDATES_PER_CUSTOMER, MIN_EVAL_CANDIDATES_PER_CUSTOMER - len(positives))
            negative_ids = _sample_unseen(catalog, excluded, required_negatives)
            assert not positives.intersection(set(negative_ids)), f"Negative candidate overlaps a positive target for customer {cid}."

            customer_features = _customer_features(customer_state, cid, date)

            for aid in positives:
                row_buffer.append(_candidate_row(customer_features, cid, aid, date, 1, split_name, article_lookup, article_columns, article_state))

            for aid in negative_ids:
                row_buffer.append(_candidate_row(customer_features, cid, aid, date, 0, split_name, article_lookup, article_columns, article_state))

            customer_progress.update(1)

            if len(row_buffer) >= batch_size:
                flush_buffer()

    customer_progress.close()
    flush_buffer()

    print(f"{split_name.capitalize()} examples built in {time.perf_counter() - started:.1f}s")

    return {
        "output_path": str(output_path),
        "row_count": row_count,
        "columns": columns,
    }


print("[1/3] Building training examples...")
_stage_started = perf_counter()
train_ex = build_train_examples(train, article_features)
print(f"Completed in {(perf_counter() - _stage_started) / 60:.1f} min | Overall: 33%")

print("[2/3] Building validation examples...")
_stage_started = perf_counter()
valid_ex = build_eval_examples(train, valid, "validation", article_features)
print(f"Completed in {(perf_counter() - _stage_started) / 60:.1f} min | Overall: 67%")

print("[3/3] Building test examples...")
_stage_started = perf_counter()
test_ex = build_eval_examples(pd.concat([train, valid], ignore_index=True), test, "test", article_features)
print(f"Completed in {(perf_counter() - _stage_started) / 60:.1f} min | Overall: 100%")

[1/3] Building training examples...


Training dates:   0%|          | 0/513 [00:00<?, ?it/s]

Train examples built in 4822.5s
Completed in 80.5 min | Overall: 33%
[2/3] Building validation examples...


Validation customers:   0%|          | 0/548527 [00:00<?, ?it/s]

Precomputing article states: 0/110 dates

Validation examples built in 2427.9s
Completed in 40.7 min | Overall: 67%
[3/3] Building test examples...


Test customers:   0%|          | 0/591464 [00:00<?, ?it/s]

Precomputing article states: 0/111 dates

Test examples built in 2510.9s
Completed in 42.7 min | Overall: 100%


In [5]:
config={'target_column':'target','time_column':'t_dat','identifier_columns':['customer_id','article_id'],'customer_features':['past_purchase_count','recent_purchase_count','historical_average_price','unique_articles_purchased','unique_categories_purchased','days_since_previous_purchase','preferred_category','preferred_garment_group','preferred_colour'],'article_features':[c for c in article_features.columns if c!='article_id'],'categorical_features':['preferred_category','preferred_garment_group','preferred_colour','product_type_name','product_group_name','garment_group_name','department_name','section_name','colour_group_name'],'numerical_features':['past_purchase_count','recent_purchase_count','historical_average_price','unique_articles_purchased','unique_categories_purchased','days_since_previous_purchase','historical_article_popularity','recent_article_popularity','historical_article_price'],'popularity_features':['historical_article_popularity','recent_article_popularity'],'excluded_leakage_features':['future purchases','future popularity','future prices','current transaction price','customer_id and article_id as numeric features'],'negative_sampling_policy':'for train, sample articles absent from customer history strictly before each positive event; for validation/test, exclude history and all target-window positives; seed 42','candidate_pool_policy':f'validation/test contain at least {MIN_EVAL_CANDIDATES_PER_CUSTOMER} negative candidates per customer when catalog permits','recent_popularity_window_days':RECENT_WINDOW_DAYS,'random_state':RANDOM_STATE,'fast_dev_run':FAST_DEV_RUN}

required_cols={'customer_id','article_id','t_dat','target'}
train_missing=required_cols.difference(train_ex['columns'])
assert not train_missing, f'train examples missing required columns: {sorted(train_missing)}'
for frame_name, metadata in [('validation',valid_ex),('test',test_ex)]:
    missing=required_cols.difference(metadata['columns'])
    assert not missing, f'{frame_name} examples missing required columns: {sorted(missing)}'

summary={'run_type':'fast_dev' if FAST_DEV_RUN else 'full','transaction_rows_loaded':int(len(tx)),'date_range':[str(tx.t_dat.min()),str(tx.t_dat.max())],'split_boundaries':{'train_end':str(train_end),'validation_end':str(valid_end)},'date_ranges':{'train':[str(train.t_dat.min()),str(train.t_dat.max())],'validation':[str(valid.t_dat.min()),str(valid.t_dat.max())],'test':[str(test.t_dat.min()),str(test.t_dat.max())]},'rows':{'train':train_ex['row_count'],'validation':valid_ex['row_count'],'test':test_ex['row_count']},'unique_customers':int(tx.customer_id.nunique()),'unique_articles':int(tx.article_id.nunique()),'negative_sampling_policy':config['negative_sampling_policy']}
(INTERIM_DIR/'hm_preference_feature_config.json').write_text(json.dumps(config,indent=2),encoding='utf-8'); (INTERIM_DIR/'hm_preference_preprocessing_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8'); print(json.dumps(summary,indent=2)); print('Conclusion: saved outputs are the disk-based inputs for 04_customer_preference.ipynb.')

{
  "run_type": "full",
  "transaction_rows_loaded": 28813419,
  "date_range": [
    "2018-09-20 00:00:00",
    "2020-09-22 00:00:00"
  ],
  "split_boundaries": {
    "train_end": "2020-02-14 00:00:00",
    "validation_end": "2020-06-03 00:00:00"
  },
  "date_ranges": {
    "train": [
      "2018-09-20 00:00:00",
      "2020-02-14 00:00:00"
    ],
    "validation": [
      "2020-02-15 00:00:00",
      "2020-06-03 00:00:00"
    ],
    "test": [
      "2020-06-04 00:00:00",
      "2020-09-22 00:00:00"
    ]
  },
  "rows": {
    "train": 60719724,
    "validation": 20261622,
    "test": 22158369
  },
  "unique_customers": 1362281,
  "unique_articles": 104547,
  "negative_sampling_policy": "for train, sample articles absent from customer history strictly before each positive event; for validation/test, exclude history and all target-window positives; seed 42"
}
Conclusion: saved outputs are the disk-based inputs for 04_customer_preference.ipynb.
